In [1]:
# Install the necessary libraries silently
!pip install -q transformers datasets sentence-transformers scikit-learn pandas

In [2]:
import pandas as pd
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, homogeneity_score

print("Loading XNLI dataset sample...")
# We are pulling a quick 1,000 sample subset of English and Turkish to get fast baseline metrics
dataset_en = load_dataset("xnli", "en", split="validation[:1000]")
dataset_tr = load_dataset("xnli", "tr", split="validation[:1000]")

df_en = pd.DataFrame(dataset_en)
df_tr = pd.DataFrame(dataset_tr)
df = pd.concat([df_en, df_tr]).reset_index(drop=True)

# Combine premise and hypothesis for embedding
df['text'] = df['premise'] + " [SEP] " + df['hypothesis']
print(f"Loaded {len(df)} total samples.")

Loading XNLI dataset sample...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

en/train-00000-of-00001.parquet:   0%|          | 0.00/50.2M [00:00<?, ?B/s]

en/test-00000-of-00001.parquet:   0%|          | 0.00/308k [00:00<?, ?B/s]

...

tr/validation-00000-of-00001.parquet:   0%|          | 0.00/172k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/392702 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5010 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2490 [00:00<?, ? examples/s]

Loaded 2000 total samples.


In [3]:
print("Loading pre-trained multilingual model...")
# This specific model is highly optimized for mapping sentences to a shared vector space
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

print("Generating embeddings (this will be very fast on your GPU)...")
embeddings = model.encode(df['text'].tolist(), show_progress_bar=True)

Loading pre-trained multilingual model...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating embeddings (this will be very fast on your GPU)...


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

In [4]:
print("Clustering with K-Means...")
# XNLI has 3 ground-truth classes: entailment, neutral, contradiction
kmeans = KMeans(n_clusters=3, random_state=42, n_init="auto")
cluster_labels = kmeans.fit_predict(embeddings)

# Evaluate the baseline
sil_score = silhouette_score(embeddings, cluster_labels)
homog_score = homogeneity_score(df['label'], cluster_labels)

print("\n--- BASELINE 2 RESULTS ---")
print(f"Silhouette Score: {sil_score:.4f}")
print(f"Homogeneity Score: {homog_score:.4f}")
print("--------------------------")

Clustering with K-Means...

--- BASELINE 2 RESULTS ---
Silhouette Score: 0.0360
Homogeneity Score: 0.0007
--------------------------
